In [7]:
data = torch.tensor(range(10))
data = data.reshape(2, 5)
data

tensor([[0, 1, 2, 3, 4],
        [5, 6, 7, 8, 9]])

In [10]:
data.size()

torch.Size([2, 5])

In [20]:
b_tensor = data.as_strided(data.permute(1,0).size(), (1,2))
b_tensor


tensor([[0, 2],
        [1, 3],
        [2, 4],
        [3, 5],
        [4, 6]])

In [21]:
ll = [0, 50, 100]

for i,j in zip(ll, ll[1:]):
    print(i,j)

0 50
50 100


In [22]:
data = torch.tensor(range(10))
offsets = [0, 5, 10]

groups = [
    data.as_strided((j - i,), (1,), i)
    for i, j in zip(offsets, offsets[1:])
]

In [23]:
groups

[tensor([0, 1, 2, 3, 4]), tensor([5, 6, 7, 8, 9])]

In [2]:
import torch
torch.__version__

'2.6.0+cu124'

In [3]:

import torch.nn as nn

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 10, bias=False)
        self.layer2 = nn.Linear(10, 10, bias=False)

    def forward(self, x):
        return self.layer2(self.layer1(x))
    
model = TinyModel()
model


TinyModel(
  (layer1): Linear(in_features=10, out_features=10, bias=False)
  (layer2): Linear(in_features=10, out_features=10, bias=False)
)

In [5]:
for name, module in model.named_children():
    print(f"子模块名称: {name}, 类型: {type(module)}")

子模块名称: layer1, 类型: <class 'torch.nn.modules.linear.Linear'>
子模块名称: layer2, 类型: <class 'torch.nn.modules.linear.Linear'>


In [6]:
import torch.nn.functional as F
class TransformerBlock(nn.Module):
    def __init__(self, d_model=10, nhead=2, dim_feedforward=5, dropout=0.1):
        super().__init__()
        # 自注意力层
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        
        # 前馈神经网络
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        
        # 归一化层
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        # 自注意力
        attn_output, _ = self.self_attn(x, x, x, attn_mask=mask)
        x = x + self.dropout(attn_output)
        x = self.norm1(x)
        
        # 前馈网络
        ff_output = self.linear2(F.gelu(self.linear1(x)))
        x = x + self.dropout(ff_output)
        x = self.norm2(x)
        
        return x
    
class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleDict({  # 显式定义的模块容器
            f"layer_{i}": TransformerBlock() for i in range(2)
        })

transformer = TransformerModel()


In [ ]:
transformer


TransformerModel(
  (layers): ModuleDict(
    (layer_0): TransformerBlock(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=10, out_features=10, bias=True)
      )
      (linear1): Linear(in_features=10, out_features=5, bias=True)
      (linear2): Linear(in_features=5, out_features=10, bias=True)
      (norm1): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layer_1): TransformerBlock(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=10, out_features=10, bias=True)
      )
      (linear1): Linear(in_features=10, out_features=5, bias=True)
      (linear2): Linear(in_features=5, out_features=10, bias=True)
      (norm1): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
      (dropou

In [12]:
import torch
from torch.nn import ModuleDict, Linear
import torch.nn as nn

class TinyDictModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 用字典保存 4 个 Linear 层
        self.tiny_block = nn.ModuleDict({
            "linear_1": nn.Linear(10, 10),
            "linear_2": nn.Linear(10, 10),
        })

    def forward(self, x):
        for lid, block in self.layers.items():
            x = block(x)
        return x

t_model = TinyDictModel()
t_model

TinyDictModel(
  (tiny_block): ModuleDict(
    (linear_1): Linear(in_features=10, out_features=10, bias=True)
    (linear_2): Linear(in_features=10, out_features=10, bias=True)
  )
)

In [ ]:
for name, param in t_model.named_parameters():
    print(f"{name:<30} shape={tuple(param.shape)}  dtype={param.dtype}  device={param.device}\n [Params]:\n{param}")

In [5]:
import torch
import torch.nn as nn

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(10, 10)       
        self.body = nn.ModuleDict({
            "0": nn.Linear(10, 10),
            "1": nn.Linear(10, 10),
        })
        self.head = nn.Linear(10, 20)              

    def forward(self, x):
        x = self.embed(x)
        for _, blk in self.layers.items():
            x = blk(x)
        return self.head(x)

tmd = TinyModel()
tmd

TinyModel(
  (embed): Embedding(10, 10)
  (body): ModuleDict(
    (0): Linear(in_features=10, out_features=10, bias=True)
    (1): Linear(in_features=10, out_features=10, bias=True)
  )
  (head): Linear(in_features=10, out_features=20, bias=True)
)

In [6]:
for name, param in tmd.named_parameters():
    print(f"-----------{name}---------------\n shape={tuple(param.shape)}  dtype={param.dtype}  device={param.device}\n [Params]:\n{param}\n------------------\n")

-----------embed.weight---------------
 shape=(10, 10)  dtype=torch.float32  device=cpu
 [Params]:
Parameter containing:
tensor([[ 1.0074,  0.3834, -0.6816,  1.1627,  0.5394, -0.0721,  1.2094,  0.9209,
          1.3692, -1.2027],
        [-0.3285, -1.1527, -1.4155,  1.9988, -1.4449,  0.5448,  0.3472, -0.2075,
          1.2941, -0.4011],
        [ 0.5608, -0.6372,  0.4711, -1.5316,  0.1333,  0.0864, -1.6930,  1.1928,
         -0.0180,  0.4965],
        [ 0.5116, -0.5575, -1.1179,  1.0397,  0.5762,  0.5008, -0.6080,  2.1619,
          1.8513, -0.3127],
        [-1.9826, -0.5681,  0.3841, -0.0331, -1.0318, -0.0538,  1.6612, -0.7654,
         -1.7254,  1.0786],
        [ 1.3673,  0.7516, -0.0576,  0.0055, -0.3171,  1.4241,  1.0915,  0.8750,
         -0.1869, -0.4452],
        [-1.0239,  1.0972, -2.2597, -2.1556, -1.3061, -0.2695, -0.2862,  0.0814,
         -1.1061, -0.5549],
        [ 0.7823,  0.5053,  2.7399, -0.0236, -1.4972,  0.1239,  0.0360, -0.5902,
          0.7869, -0.0850],
       

In [ ]:
import os
import torch
import torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed._composable.fsdp import fully_shard, MixedPrecisionPolicy
# 设置单进程模拟环境
os.environ['RANK'] = '0'
os.environ['WORLD_SIZE'] = '2'
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12355'
dist.destroy_process_group()
# 初始化进程组
dist.init_process_group(backend='gloo',rank=0, world_size=2) 

cpu_local_mesh = init_device_mesh("cpu", mesh_shape=(2,))

mp_policy = MixedPrecisionPolicy(
            param_dtype=torch.bfloat16
        )



In [ ]:
for layer_id, transformer_block in tmd.layers.items():
    if config.train.reshard_after_forward:
        reshard_after_forward = int(layer_id) < len(model.layers) - 1
    else:
        reshard_after_forward = False
    fully_shard(
        transformer_block,
        mp_policy=mp_policy,
        mesh=elastic_device_mesh.cuda_local_mesh,
        reshard_after_forward=reshard_after_forward,
        offload_policy=offload_policy,
    )


In [ ]:
fully_shard(
    model,
    mp_policy=mp_policy,
    mesh=elastic_device_mesh.cuda_local_mesh,
    reshard_after_forward=config.train.reshard_after_forward,
    offload_policy=offload_policy,
)